# MNIST: Data Preparation and Exploration
**Goal:** Load MNIST through Scikit-learn, normalize and flatten the images, then explore the data to understand its distribution and characteristics.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml, load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
np.random.seed(42)

## 2. Load the MNIST dataset
`fetch_openml` pulls MNIST through Scikit-learn (no manual download). The first run takes a minute; after that it loads from the local cache.

In [ ]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")

X_raw = mnist.data          # pixel values 0-255
y = mnist.target.astype(int)  # labels come as strings -> convert to int

print("X shape:", X_raw.shape)
print("y shape:", y.shape)
print("X dtype:", X_raw.dtype, "| min:", X_raw.min(), "| max:", X_raw.max())

## 3. Preprocessing

### 3.1 Check data quality
Confirm there are no missing values and that pixel values sit in the expected 0-255 range.

In [ ]:
print("Missing values in X:", np.isnan(X_raw).sum())
print("Missing values in y:", pd.isna(y).sum())

assert X_raw.shape == (70000, 784), f"Unexpected shape: {X_raw.shape}"
assert X_raw.min() >= 0 and X_raw.max() <= 255, "Pixel values outside 0-255"
print("Data quality checks passed.")

### 3.2 Normalize
Divide by 255 so every pixel is between 0 and 1. This keeps features on the same scale, which helps most ML algorithms train faster and more reliably.

In [ ]:
X = (X_raw / 255.0).astype(np.float32)
print("After normalization -> min:", X.min(), "| max:", X.max(), "| dtype:", X.dtype)

### 3.3 Flatten
Each image is 28x28 pixels. `fetch_openml` already returns them flattened into 784-length vectors. The cell below shows the round trip between the image form and the vector form.

In [ ]:
images = X.reshape(-1, 28, 28)   # image form (for plotting)
X_flat = images.reshape(len(images), -1)  # vector form (for analysis)

print("Image form: ", images.shape)
print("Vector form:", X_flat.shape)
assert np.array_equal(X_flat, X), "Flattening changed the data"
X = X_flat

### 3.4 Train / test split
MNIST has a standard split: the first 60,000 images are for training and the last 10,000 for testing.

In [ ]:
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)

## 4. Exploratory Data Analysis

### 4.1 Class distribution

In [ ]:
train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts = pd.Series(y_test).value_counts().sort_index()

dist = pd.DataFrame({
    "train_count": train_counts,
    "train_pct": (train_counts / len(y_train) * 100).round(2),
    "test_count": test_counts,
    "test_pct": (test_counts / len(y_test) * 100).round(2),
})
dist.index.name = "digit"
display(dist)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=train_counts.index, y=train_counts.values, ax=axes[0], color="steelblue")
axes[0].set_title("Training set: images per digit")
sns.barplot(x=test_counts.index, y=test_counts.values, ax=axes[1], color="darkorange")
axes[1].set_title("Test set: images per digit")
for ax in axes:
    ax.set_xlabel("Digit"); ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=120)
plt.show()

**Observation:** The classes are roughly balanced (each digit is about 9-11% of the data). Digit 1 is the most common and 5 the least common, but the gap is small, so no rebalancing is needed.

### 4.2 Sample images

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for digit in range(10):
    idx = np.random.choice(np.where(y_train == digit)[0], 10, replace=False)
    for j, i in enumerate(idx):
        ax = axes[digit, j]
        ax.imshow(X_train[i].reshape(28, 28), cmap="gray")
        ax.axis("off")
plt.suptitle("10 random samples per digit (rows = 0-9)", fontsize=14)
plt.tight_layout()
plt.savefig("sample_images.png", dpi=120)
plt.show()

**Observation:** Handwriting varies a lot within each digit (slant, thickness, size), which is what makes this a meaningful classification problem.

### 4.3 Average image per digit

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for digit in range(10):
    mean_img = X_train[y_train == digit].mean(axis=0).reshape(28, 28)
    axes[digit].imshow(mean_img, cmap="hot")
    axes[digit].set_title(str(digit))
    axes[digit].axis("off")
plt.suptitle("Mean image per digit")
plt.tight_layout()
plt.savefig("mean_images.png", dpi=120)
plt.show()

**Observation:** The averages are clearly recognizable, so each digit has a consistent overall shape. Blurrier averages (like 4, 5, 8) suggest more variation in how people write them.

### 4.4 Pixel intensity distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X_train.ravel(), bins=50, color="steelblue")
axes[0].set_title("All pixel values")
axes[0].set_yscale("log")
axes[0].set_xlabel("Normalized intensity (0-1)")

nonzero = X_train[X_train > 0]
axes[1].hist(nonzero, bins=50, color="darkorange")
axes[1].set_title("Non-zero pixel values only")
axes[1].set_xlabel("Normalized intensity (0-1)")
plt.tight_layout()
plt.savefig("pixel_distribution.png", dpi=120)
plt.show()

print(f"Share of pixels that are exactly 0: {(X_train == 0).mean():.1%}")

**Observation:** Most pixels are background (value 0), roughly 80%. Among the 'ink' pixels, values cluster near 1 (fully white strokes), with a smaller spread of mid-tones along stroke edges.

### 4.5 Which pixels carry information?

In [ ]:
pixel_var = X_train.var(axis=0).reshape(28, 28)
always_zero = (X_train.max(axis=0) == 0).sum()

plt.figure(figsize=(5, 4))
sns.heatmap(pixel_var, cmap="viridis", xticklabels=False, yticklabels=False)
plt.title("Variance of each pixel position")
plt.tight_layout()
plt.savefig("pixel_variance.png", dpi=120)
plt.show()

print(f"Pixels that are 0 in every training image: {always_zero} of 784")

**Observation:** The border pixels almost never change, and dozens are always 0. Those features add nothing, which hints that the data can be compressed into fewer dimensions (see PCA below).

### 4.6 Ink per digit

In [ ]:
ink = pd.DataFrame({"digit": y_train, "ink": X_train.sum(axis=1)})

plt.figure(figsize=(10, 4))
sns.boxplot(data=ink, x="digit", y="ink", color="lightsteelblue")
plt.title("Total ink (sum of pixel values) per digit")
plt.xlabel("Digit"); plt.ylabel("Total ink")
plt.tight_layout()
plt.savefig("ink_per_digit.png", dpi=120)
plt.show()

display(ink.groupby("digit")["ink"].describe().round(1))

**Observation:** Digit 1 uses the least ink and 0 and 8 use the most. Ink alone can't separate all digits, but it's a simple feature that already carries some signal.

### 4.7 Dimensionality check with PCA

In [ ]:
pca_full = PCA().fit(X_train)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

for target in [0.80, 0.90, 0.95, 0.99]:
    n = np.argmax(cum_var >= target) + 1
    print(f"{int(target*100)}% of variance kept with {n} components")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 785), cum_var)
plt.axhline(0.95, color="red", linestyle="--", label="95%")
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance")
plt.legend()
plt.tight_layout()
plt.savefig("pca_variance.png", dpi=120)
plt.show()

In [ ]:
# 2D projection on a 5,000-image sample so the plot stays readable
idx = np.random.choice(len(X_train), 5000, replace=False)
X_2d = PCA(n_components=2).fit_transform(X_train[idx])

plt.figure(figsize=(8, 6))
sc = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_train[idx], cmap="tab10", s=5, alpha=0.7)
plt.colorbar(sc, ticks=range(10), label="Digit")
plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.title("MNIST projected onto 2 principal components")
plt.tight_layout()
plt.savefig("pca_2d.png", dpi=120)
plt.show()

**Observation:** About 150 components keep 95% of the variance, so the 784 pixels are highly redundant. In 2D, some digits (like 0 and 1) form distinct clusters while others overlap heavily, which shows a linear 2D view isn't enough to separate every class.

## 5. Optional: Scikit-learn's built-in `digits` dataset
`load_digits` ships with Scikit-learn itself (no download at all). It's a smaller, lower-resolution version of the same idea: 1,797 images at 8x8 pixels.

In [ ]:
digits = load_digits()
Xd = digits.data / 16.0   # pixel values are 0-16 here
yd = digits.target

print("digits X shape:", Xd.shape, "| classes:", np.unique(yd))

fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for d in range(10):
    axes[d].imshow(Xd[yd == d][0].reshape(8, 8), cmap="gray")
    axes[d].set_title(str(d)); axes[d].axis("off")
plt.suptitle("load_digits samples (8x8)")
plt.tight_layout()
plt.savefig("digits_samples.png", dpi=120)
plt.show()

comparison = pd.DataFrame({
    "MNIST": ["70,000", "28x28", 784, "0-255"],
    "load_digits": ["1,797", "8x8", 64, "0-16"],
}, index=["Images", "Resolution", "Features", "Raw pixel range"])
display(comparison)

## 6. Summary of findings
- **Size:** 70,000 grayscale images (60,000 train / 10,000 test), each flattened to 784 features.
- **Quality:** No missing values; pixels were normalized from 0-255 to 0-1.
- **Balance:** All 10 digits are roughly equally represented, so accuracy is a fair metric and no resampling is needed.
- **Sparsity:** About 80% of pixels are background, and border pixels carry almost no information.
- **Redundancy:** PCA keeps 95% of the variance with about 150 of 784 components, so dimensionality reduction is a sensible next step.
- **Separability:** Some digits separate easily (0, 1) while others overlap (4/9, 3/5/8), which points to where a classifier is most likely to make mistakes.